# DL & GenAI project milestone - 2 (23f3001252)

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Load datasets

In [3]:
## Load datasets
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [4]:
print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)
print("Sample Shape:", sample.shape)

Train Shape: (2000, 8)
Test Shape: (500, 7)
Sample Shape: (500, 2)


In [5]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


## Question 1
#### Q1. Load train.csv using Hugging Face datasets, create combined_text, and find the character length of row 51

In [7]:
# Install datasets if not already installed

!pip install datasets

In [8]:
## Q1. Load train.csv using Hugging Face datasets, create combined_text, and find the character length of row 51

from datasets import load_dataset

# Load train.csv using Hugging Face datasets
dataset = load_dataset("csv", data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"})

train_ds = dataset["train"]

# Create combined_text = prompt + " " + A
train_ds = train_ds.map(
    lambda example: {
        "combined_text": example["prompt"] + " " + example["A"]
    }
)

# Character length (NOT tokens) for row index 51
length = len(train_ds[51]["combined_text"])

print("Combined text:")
print(train_ds[51]["combined_text"])
print("\nCharacter length:", length)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Combined text:
Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.

Character length: 614


## Question 2
#### Initialize the bert-base-uncased tokenizer and find the vocabulary size

In [9]:
!pip install transformers

In [10]:
### Initialize the bert-base-uncased tokenizer and find the vocabulary size
# Install transformers if needed

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Vocabulary Size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary Size: 30522


## Question - 3
#### Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("SEP Token:", tokenizer.sep_token)
print("SEP Token ID:", tokenizer.sep_token_id)

SEP Token: [SEP]
SEP Token ID: 102


## Question - 4
 Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor? 

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

encoded = tokenizer(
    train_df["prompt"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("input_ids shape:", encoded["input_ids"].shape)

input_ids shape: torch.Size([2000, 128])


## Question - 5

In [13]:
hidden_size = 768
num_attention_heads = 12

head_dim = hidden_size // num_attention_heads

print("Dimension of each attention head:", head_dim)

Dimension of each attention head: 64


## Question - 6

In [14]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Row 0 prompt
text = train_ds[0]["prompt"]

# Default tokenizer settings
inputs = tokenizer(
    text,
    return_tensors="pt"
)

outputs = model(**inputs)

print("last_hidden_state shape:", outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


last_hidden_state shape: torch.Size([1, 31, 768])


## Question - 7

In [15]:
cls_embedding = outputs.last_hidden_state[0, 0]

print("CLS embedding shape:", cls_embedding.shape)

sum_first5 = cls_embedding[:5].sum().item()

print("Sum of first 5 values:", round(sum_first5, 4))

CLS embedding shape: torch.Size([768])
Sum of first 5 values: -1.2001


## Question - 8

In [18]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

outputs = model(**inputs)

# Show tokenization
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(tokens)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']


In [19]:
fusion_index = tokens.index("fusion")

print("Fusion token index:", fusion_index)

Fusion token index: 4


In [20]:
last_layer_attention = outputs.attentions[-1]

print(last_layer_attention.shape)

torch.Size([1, 12, 10, 10])


In [21]:
attention_matrix = last_layer_attention[0, 0]

cls_to_fusion = attention_matrix[0, fusion_index].item()

print("Attention weight:", round(cls_to_fusion, 4))

Attention weight: 0.1025


## Question - 9

In [22]:
# !pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer, util

# Load MiniLM model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Row 0
prompt = train_ds[0]["prompt"]
option_b = train_ds[0]["B"]

# Generate embeddings
embeddings = model.encode([prompt, option_b], convert_to_tensor=True)

# Cosine similarity
similarity = util.cos_sim(embeddings[0], embeddings[1])

print("Cosine Similarity:", round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cosine Similarity: 0.7658


## Question - 10

In [23]:
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm.auto import tqdm

In [24]:
def mapk(actual, predicted, k=3):
    score = 0.0

    for a, p in zip(actual, predicted):
        try:
            rank = p[:k].index(a)
            score += 1.0 / (rank + 1)
        except ValueError:
            pass

    return score / len(actual)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [25]:
### TF-idf- Pipeline - 1

tfidf_predictions = []

for row in tqdm(train_ds):

    prompt = row["prompt"]
    options = [row[c] for c in ["A","B","C","D","E"]]

    docs = [prompt] + options

    vectorizer = TfidfVectorizer()

    tfidf = vectorizer.fit_transform(docs)

    similarities = cosine_similarity(
        tfidf[0:1],
        tfidf[1:]
    )[0]

    ranking = np.argsort(similarities)[::-1]

    labels = ["A","B","C","D","E"]

    tfidf_predictions.append(
        [labels[i] for i in ranking[:3]]
    )

  0%|          | 0/2000 [00:00<?, ?it/s]

In [26]:
#### Pipeline 2 — MiniLM

# Load the model once.

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
minilm_predictions = []

for row in tqdm(train_ds):

    prompt = row["prompt"]

    options = [row[c] for c in ["A","B","C","D","E"]]

    embeddings = model.encode(
        [prompt] + options,
        convert_to_tensor=True
    )

    prompt_embedding = embeddings[0]

    option_embeddings = embeddings[1:]

    similarities = util.cos_sim(
        prompt_embedding,
        option_embeddings
    ).cpu().numpy()[0]

    ranking = np.argsort(similarities)[::-1]

    labels = ["A","B","C","D","E"]

    minilm_predictions.append(
        [labels[i] for i in ranking[:3]]
    )

  0%|          | 0/2000 [00:00<?, ?it/s]

In [28]:
### Compute MAP@3
actual = train_ds["answer"]

tfidf_map3 = mapk(actual, tfidf_predictions)

minilm_map3 = mapk(actual, minilm_predictions)

print("TF-IDF MAP@3 :", round(tfidf_map3, 5))
print("MiniLM MAP@3 :", round(minilm_map3, 5))

TF-IDF MAP@3 : 0.32692
MiniLM MAP@3 : 0.42308


In [29]:
### Count improvements
### This answers the second part.

count = 0

for actual_answer, tfidf_pred, minilm_pred in zip(
    actual,
    tfidf_predictions,
    minilm_predictions,
):

    if (
        actual_answer not in tfidf_pred
        and actual_answer in minilm_pred
    ):
        count += 1

print("Improved Questions:", count)

Improved Questions: 564


In [30]:
print(len(train_ds))

2000


## Question - 11

In [31]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row = train_ds[1]

sequence = row["prompt"]

candidate_labels = [
    row["A"],
    row["B"],
    row["C"]
]

result = classifier(
    sequence,
    candidate_labels=candidate_labels
)

print(result)

print("\nTop Label:", result["labels"][0])
print("Top Score:", round(result["scores"][0], 4))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

## Question - 12

In [32]:
result_sigmoid = classifier(
    sequence,
    candidate_labels=candidate_labels,
    multi_label=True
)

print(result_sigmoid)

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion 

In [33]:
softmax_sum = sum(result["scores"])

sigmoid_sum = sum(result_sigmoid["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("Softmax Sum :", softmax_sum)
print("Sigmoid Sum :", sigmoid_sum)
print("Absolute Difference:", round(difference, 4))

Softmax Sum : 0.9999999701976776
Sigmoid Sum : 0.0005096072964079212
Absolute Difference: 0.9995


## Question - 13

In [36]:
### FLAN-T5 Small (Generative AI)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

row = train_ds[0]

prompt = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} "
    f"or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

B


In [38]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

In [39]:
output = generator(
    prompt,
    max_new_tokens=5
)

print(output)
print("\nGenerated Text:", output[0]["generated_text"])


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B."}]

Generated Text: Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct a